In [6]:
!uv add google-adk google-generativeai -q
!uv add plotly pandas python-dotenv -q

In [8]:
# Default libs
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid5
from typing import Any, List

# Installed libs
import pandas as pd
import plotly.graph_objects as go
import vertexai
from IPython.display import HTML, Markdown, display

# ADK libs
import google.adk as adk
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.adk.tools import google_search
from google.genai import types
from google.genai.types import Content, Part

# Load envs
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
print(os.getenv("GOOGLE_CLOUD_PROJECT"))

project-de34b621-684f-4b33-b42


In [10]:
GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")

In [15]:
# Add gcloud to PATH for Jupyter
import os
import shutil

# Find gcloud executable
gcloud_path = shutil.which("gcloud")
if gcloud_path:
    gcloud_dir = os.path.dirname(gcloud_path)
    os.environ["PATH"] = gcloud_dir + os.pathsep + os.environ.get("PATH", "")
    print(f"Found gcloud at: {gcloud_path}")
else:
    print("gcloud not found in PATH, trying default locations...")
    # Try common locations
    default_paths = [
        os.path.expanduser("~/.local/bin/gcloud"),
        os.path.expanduser("~/google-cloud-sdk/bin/gcloud"),
        "/usr/local/bin/gcloud",
    ]
    for path in default_paths:
        if os.path.exists(path):
            os.environ["PATH"] = os.path.dirname(path) + os.pathsep + os.environ.get("PATH", "")
            print(f"Found gcloud at: {path}")
            break

gcloud not found in PATH, trying default locations...
Found gcloud at: /home/vasim/google-cloud-sdk/bin/gcloud


In [ ]:
# !gcloud auth login
# !gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=PLVML6PDRYs1Y0bJZPpJe6LqSJWUAx&access_type=offline&code_challenge=2gwTf3Uc9XpPxFJHrZG5N1aEFzvhBVatkn-cvpM_Ir0&code_challenge_method=S256

Gtk-Message: 20:28:32.499: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.

Credentials saved to file: [/home/vasim/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "project-de34b621-684f-4b33-b42" was added to ADC which can be used by Google client librarie

In [ ]:
# !gcloud config set project $GOOGLE_CLOUD_PROJECT

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [ ]:
# !gcloud services enable aiplatform.googleapis.com --project=$GOOGLE_CLOUD_PROJECT

Install gcloud cli if it is not available:
```shell
# Download and install gcloud SDK
curl https://sdk.cloud.google.com | bash
source ~/.bashrc
```

---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!